[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-5-3-unit-testing-with-pytest.ipynb)

# Lesson 5.3 · Unit Testing with pytest

**Free Python For Development** · Topic 5: Error Handling, Debugging & Testing

_Fixtures, parametrize, mocking._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-5-3)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q pytest requests

## Introduction & Core Concepts of Unit Testing with pytest

Let’s talk about **unit testing** in Python using **pytest**. Imagine you are building a bridge; you wouldn’t just hope it holds — you would test each part to make sure it is strong. That is exactly what unit testing does for your code.

With pytest you write simple test functions that check whether your functions behave as expected. This prevents bugs and keeps your code reliable as it grows.

> 💡 **Key insight:** pytest discovers any function whose name starts with `test_` (or any class whose name starts with `Test`) and runs it automatically. No boilerplate test runner is required.

### A Minimal Example

In [ ]:
%%writefile math_utils.py
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

In [ ]:
%%writefile test_math_utils.py
import pytest
from math_utils import add, divide

def test_add():
    assert add(2, 3) == 5
    assert add(-1, 1) == 0

def test_divide():
    assert divide(10, 2) == 5.0

def test_divide_by_zero():
    with pytest.raises(ValueError, match="zero"):
        divide(10, 0)

In [ ]:
!python -m pytest -v test_math_utils.py

**What pytest gives you**

- Automatic test discovery.
- Clear assertion introspection (you see the actual values on failure).
- Built-in support for expected exceptions via `pytest.raises`.
- Rich plugin ecosystem and excellent reporting.

> 💡 **Mental model:** Each test is a small experiment that asks “Does this piece of code still behave the way I expect?” Running the whole suite is like inspecting every bolt on the bridge before traffic is allowed.

## Fixtures & Parametrize — Reusable Setup and Multiple Inputs

Pytest offers **fixtures** — reusable setup and teardown code that runs before (and after) your tests. If a test needs a temporary directory, a database connection, or a sample data structure, a fixture can create it once and clean it up automatically.

Another powerful feature is **parametrize**. Instead of writing nearly identical tests for different input values, you declare the inputs once and let pytest run the test for every combination.

**Fixtures**

Functions decorated with `@pytest.fixture`. Tests request them by name as parameters. Scope can be function, class, module or session.

**Parametrize**

`@pytest.mark.parametrize` turns one test function into many test cases, each with different arguments.

### Fixtures + Parametrize in Action

In [ ]:
%%writefile test_cart.py
import pytest

@pytest.fixture
def sample_cart():
    """Provide a fresh shopping cart for each test."""
    return {"items": [], "total": 0.0}

def add_item(cart, name, price):
    cart["items"].append(name)
    cart["total"] += price
    return cart

def test_add_item(sample_cart):
    add_item(sample_cart, "Book", 12.99)
    assert len(sample_cart["items"]) == 1
    assert sample_cart["total"] == 12.99

@pytest.mark.parametrize("a, b, expected", [
    (2, 3, 5),
    (0, 0, 0),
    (-1, 1, 0),
    (10, -3, 7),
])
def test_add_parametrized(a, b, expected):
    assert a + b == expected

In [ ]:
!python -m pytest -v test_cart.py

**Why these features matter**

- Fixtures eliminate duplicated setup code.
- Parametrize keeps tests DRY while covering many cases.
- Each parameter set appears as a separate, clearly named test result.
- Failures point to the exact input that broke the assertion.

> 💡 **Architecture tip:** Keep fixtures focused. Prefer many small fixtures that compose rather than one giant fixture that does everything.

## Mocking External Dependencies & Best Practices

Real code often talks to the outside world — HTTP APIs, databases, the file system, or the system clock. Those dependencies can be slow, flaky, or expensive. **Mocking** lets you replace them with controlled fakes so your tests stay fast, deterministic and focused on your own logic.

### Mocking with unittest.mock / pytest-mock

In [ ]:
%%writefile test_user_api.py
from unittest.mock import patch, Mock
import requests

def get_user_name(user_id):
    response = requests.get(f"https://api.example.com/users/{user_id}")
    response.raise_for_status()
    return response.json()["name"]

def test_get_user_name():
    fake_response = Mock()
    fake_response.json.return_value = {"name": "Alice"}
    fake_response.raise_for_status.return_value = None

    with patch("requests.get", return_value=fake_response) as mock_get:
        name = get_user_name(42)
        assert name == "Alice"
        mock_get.assert_called_once_with(
            "https://api.example.com/users/42"
        )

In [ ]:
!python -m pytest -v test_user_api.py

**Production habits**

- Test behavior, not implementation details.
- Prefer real objects when they are fast and deterministic.
- Mock only at the boundary (network, disk, time).
- Keep tests independent — order should never matter.
- Run the suite on every commit (CI).

**What to test**

Happy paths, edge cases, error handling, and any public contract your function promises to callers.

**What not to over-test**

Private helpers, third-party library internals, or trivial getters/setters that add no value.

> 💡 **Big takeaway:** pytest makes unit testing easier, faster and more robust. Fixtures give you clean setup, parametrize multiplies coverage without duplication, and mocking isolates your logic from the outside world. Testing early and often catches problems before they reach users and makes both the code and the developer’s life more reliable.

---

✅ **Lesson complete!** Next up: **Lesson 5.4 · TDD & CI/CD** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-5-4) or open `lesson-5-4-tdd-and-ci-cd.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).